In [1]:
import os
import torch
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images

device = "cuda" if torch.cuda.is_available() else "cpu"
# bfloat16 is supported on Ampere GPUs (Compute Capability 8.0+) 
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

# Initialize the model and load the pretrained weights.
# This will automatically download the model weights the first time it's run, which may take a while.
model = VGGT.from_pretrained("facebook/VGGT-1B").to(device)

/root/miniconda3/envs/vggt/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [35]:
import time
import os
import torch

datapath = "datasets/tum/rgbd_dataset_freiburg1_desk2/rgb"
img_list = os.listdir(datapath)
ori_image_names = img_list[4:15]
print(len(ori_image_names))
image_names = [os.path.join(datapath, img) for img in ori_image_names]

images = load_and_preprocess_images(image_names).to(device)

# Assume images shape is (1, N, C, H, W)
N, C, H, W = images.shape
print(images.shape)

start_time = time.time()
with torch.no_grad():
    with torch.cuda.amp.autocast(dtype=dtype):
        images = images[None]  # add batch dimension
        predictions = model(images)

end_time = time.time()
print(f"Spent time: {end_time - start_time:.4f} seconds")

11
torch.Size([11, 3, 392, 518])
Spent time: 1.2894 seconds
